# solving using astrometry.net

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name=version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [1]:
import importlib, sys, subprocess
packages = "numpy, matplotlib, astropy, astroquery, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        #print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")

%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

**** module numpy is installed
**** module matplotlib is installed
**** module astropy is installed
**** module astroquery is installed
**** module version_information is installed
This notebook was generated at 2023-09-25 02:11:45 (KST = GMT+0900) 
0 Python     3.11.4 64bit [GCC 11.2.0]
1 IPython    8.12.2
2 OS         Linux 5.15.0 84 generic x86_64 with glibc2.31
3 numpy      1.25.2
4 matplotlib 3.7.2
5 astropy    5.2.1
6 astroquery 0.4.6
7 version_information 1.0.4


### import modules

In [6]:
from glob import glob
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.stats import sigma_clip
from ccdproc import combine, ccd_process, CCDData

import ysfitsutilpy as yfu
import ysphotutilpy as ypu

import _astro_utilities
import _Python_utilities

plt.rcParams.update({'figure.max_open_warning': 0})

In [7]:
#%%
#######################################################
# read all files in base directory for processing
BASEDIR = Path("/mnt/Rdata/OBS_data") 

DOINGDIR = Path(BASEDIR / "ccd_test_folder")

#DOINGDIRs = sorted(_Python_utilities.getFullnameListOfallfilessubDirs(DOINGDIR))
#print ("len(DOINGDIRs): ", len(DOINGDIRs))
#######################################################

In [8]:
fpaths = sorted(list(DOINGDIR.glob("*.fit")))
fpaths

[PosixPath('/mnt/Rdata/OBS_data/ccd_test_folder/M27_LIGHT_H_2016-10-05-12-25-13_500sec_TMB130ss-x75_QSI683ws_-20c_1bin.fit'),
 PosixPath('/mnt/Rdata/OBS_data/ccd_test_folder/M27_LIGHT_H_2016-10-05-12-34-05_500sec_TMB130ss-x75_QSI683ws_-20c_1bin.fit'),
 PosixPath('/mnt/Rdata/OBS_data/ccd_test_folder/M27_LIGHT_H_2016-10-05-13-01-45_500sec_TMB130ss-x75_QSI683ws_-20c_1bin.fit'),
 PosixPath('/mnt/Rdata/OBS_data/ccd_test_folder/M27_LIGHT_O_2016-10-05-13-37-16_500sec_TMB130ss-x75_QSI683ws_-20c_1bin.fit'),
 PosixPath('/mnt/Rdata/OBS_data/ccd_test_folder/M27_LIGHT_O_2016-10-05-13-46-08_500sec_TMB130ss-x75_QSI683ws_-20c_1bin.fit'),
 PosixPath('/mnt/Rdata/OBS_data/ccd_test_folder/M27_LIGHT_O_2016-10-05-13-55-01_500sec_TMB130ss-x75_QSI683ws_-20c_1bin.fit'),
 PosixPath('/mnt/Rdata/OBS_data/ccd_test_folder/M27_LIGHT_O_2016-10-05-14-03-53_500sec_TMB130ss-x75_QSI683ws_-20c_1bin.fit'),
 PosixPath('/mnt/Rdata/OBS_data/ccd_test_folder/M27_LIGHT_O_2016-10-05-14-12-46_500sec_TMB130ss-x75_QSI683ws_-20c_1bin

In [16]:
import shutil
from astroquery.astrometry_net import AstrometryNet

from astropy.io import fits
ast = AstrometryNet()
ast.api_key = 'bldvwzzuvktnwfph'

In [17]:
solve_timeout = 600
fpath = fpaths[0]
print(fpath)
wcs_header = ast.solve_from_image(str(fpath),
                        force_image_upload=True,
                        solve_timeout = solve_timeout,
                        )
print("wcs_header :", wcs_header)

/mnt/Rdata/OBS_data/ccd_test_folder/M27_LIGHT_H_2016-10-05-12-25-13_500sec_TMB130ss-x75_QSI683ws_-20c_1bin.fit
Solving

In [14]:
dir(wcs_header)

NameError: name 'wcs_header' is not defined

In [17]:
#fpath = Path(fpaths[0])
fpath = fpaths[5]
print(fpath)

submission_id = None
solve_timeout = 600

hdul = fits.open(str(fpath))
if not 'B_1_1' in hdul[0].header :
    print("it's not solved file...")
    try_again = True    

while try_again:
    try:
        if not submission_id:
            wcs_header = ast.solve_from_image(str(fpath),
                                force_image_upload=True,
                                solve_timeout = solve_timeout,
                                submission_id=submission_id)
        else:
            wcs_header = ast.monitor_submission(submission_id,
                                                solve_timeout = solve_timeout)
    except TimeoutError as e:
        submission_id = e.args[1]
    else:
        # got a result, so terminate
        try_again = False

if wcs_header:
    # Code to execute when solve succeeds
    print("fits file solved successfully...")
    shutil.copy(str(fpath), str(fpath.parents[0] / f"{fpath.stem}.tmp"))

    with fits.open(str(fpath.parents[0] / f"{fpath.stem}.tmp"), mode='update') as filehandle:
        for card in wcs_header :
            try: 
                print(card, wcs_header[card], wcs_header.comments[card])
                filehandle[0].header.set(card, wcs_header[card], wcs_header.comments[card])
            except : 
                print(card)
        filehandle.flush


    #shutil.move(str(fpath.parents[0] / f"{fpath.stem}.tmp"), str(fpath.parents[0] / f"{fpath.stem}.fits"))
    #print(str(fpath.parents[0] / f'{fpath.stem}.fits')+" is created...")
else:
    # Code to execute when solve fails
    print("fits file solving failure...")


/mnt/Rdata/OBS_data/ccd_test_folder/M27_LIGHT_O_2016-10-05-13-55-01_500sec_TMB130ss-x75_QSI683ws_-20c_1bin.fit
it's not solved file
Solving